# Tesseract Layout-Aware OCR Benchmark
**Pierce 1890 Medical Adviser · Team G07 · A2 OCR benchmarking**

## Strategy
Running Tesseract on the full page gives inflated error because it reads figures, rules and plate captions as body text.  
This notebook instead:
1. Loads **Chandra `chunks.jsonl`** layout blocks (text regions only — Image/Figure/Diagram excluded)
2. Renders each page at **300 DPI** with PyMuPDF
3. **Crops each text region** and runs Tesseract on the crop individually
4. Reassembles the page transcript in reading order
5. Scores against hand-verified **ground-truth labels** (`labels.jsonl`)
6. Also scores **Chandra's own text** against the same GT — so you see the gap

## Kaggle datasets required
| Dataset | Slug |
|---|---|
| Pierce PDF | `kmazd1110/dl-peoples-common-sense-med-advisor` |
| Chandra layout blocks | `cruelangelssprint/pierce-1890-figure-and-ocr-outputs` |
| Ground-truth labels | `kmazd1110/pierce-book-gt` |

Add all three under **+ Add Data** before running.

## Cell 1 — Install system & Python dependencies

In [ ]:
import subprocess, sys

# Tesseract OCR (system package)
subprocess.run(["apt-get", "install", "-y", "-q", "tesseract-ocr", "tesseract-ocr-eng"], check=True)

# Python wrappers
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pytesseract", "pymupdf", "jiwer", "opencv-python-headless"], check=True)

print("All dependencies installed.")

## Cell 2 — Imports & output directory

In [ ]:
import csv, json, re, time
from collections import defaultdict
from pathlib import Path

import cv2
import fitz          # PyMuPDF
import numpy as np
import pytesseract
from jiwer import cer as compute_cer, wer as compute_wer

OUT_DIR = Path("/kaggle/working/tesseract_layout_bench")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Imports OK. Output directory:", OUT_DIR)

## Cell 3 — Paths & configuration

> Edit the paths below if your dataset slugs differ from the defaults.

In [ ]:
# ── Kaggle input paths ────────────────────────────────────────────────────────
PDF_PATH = Path(
    "/kaggle/input/datasets/kmazd1110/dl-peoples-common-sense-med-advisor"
    "/EN_The-Peoples-Common-Sense-Medical-Adviser.pdf"
)
CHANDRA_PATH = Path(
    "/kaggle/input/datasets/cruelangelssprint/pierce-1890-figure-and-ocr-outputs"
    "/chandra/chunks.jsonl"
)
LABELS_PATH = Path("/kaggle/input/datasets/kmazd1110/pierce-book-gt/labels.jsonl")

# ── OCR settings ──────────────────────────────────────────────────────────────
DPI          = 300        # render resolution (300 DPI = publication-quality)
TESSERACT_CFG = "--psm 6"  # PSM 6 = single uniform block per crop

# ── Verify files exist before doing anything expensive ───────────────────────
for p in [PDF_PATH, CHANDRA_PATH, LABELS_PATH]:
    status = "OK" if p.exists() else "MISSING"
    print(f"[{status}] {p}")
    assert p.exists(), f"File not found: {p}"

## Cell 4 — Helper functions

In [ ]:
import re as _re

def _chandra_label_kind(label) -> str:
    """
    Text-bearing labels → 'text'  (OCR these)
    Visual/structural   → 'skip'  (do not OCR)
    """
    TEXT_LABELS = {
        "text", "section-header", "caption",
        "footnote", "list-group", "table",
    }
    if label is None:
        return "skip"
    return "text" if str(label).lower().strip() in TEXT_LABELS else "skip"


def _strip_html(html: str) -> str:
    """Strip HTML tags from Chandra content field."""    return _re.sub(r"<[^>]+>", " ", html).strip()


def render_page(doc: fitz.Document, page_idx: int, dpi: int = DPI) -> np.ndarray:
    """Render a PDF page to a numpy BGR image at the given DPI."""
    pix = doc[page_idx].get_pixmap(dpi=dpi)
    arr = np.frombuffer(pix.samples, np.uint8).reshape(pix.height, pix.width, pix.n)
    return cv2.cvtColor(arr, cv2.COLOR_RGB2BGR if pix.n == 3 else cv2.COLOR_RGBA2BGR)


def bbox_to_pixel(
    bbox: list, page_box: list, img_w: int, img_h: int
) -> tuple[int, int, int, int] | None:
    """
    Convert Chandra bbox [x0,y0,x1,y1] (absolute pixels in page_box space)
    into pixel coordinates of our rendered image (img_w x img_h).

    Chandra page_box schema: [pb_x0, pb_y0, pb_x1, pb_y1]
      → page width  = pb_x1 - pb_x0
      → page height = pb_y1 - pb_y0

    Returns None if page dimensions are zero (degenerate block — skip it).
    """
    pb_x0, pb_y0, pb_x1, pb_y1 = (float(v) for v in page_box)
    cw = pb_x1 - pb_x0   # page width in Chandra image pixels
    ch = pb_y1 - pb_y0   # page height in Chandra image pixels
    if cw <= 0.0 or ch <= 0.0:
        return None  # degenerate page_box — skip

    x0, y0, x1, y1 = (float(v) for v in bbox)
    # Normalise to [0,1] relative to page_box, then scale to rendered image
    px0 = max(0,      int((x0 - pb_x0) / cw * img_w))
    py0 = max(0,      int((y0 - pb_y0) / ch * img_h))
    px1 = min(img_w,  int((x1 - pb_x0) / cw * img_w))
    py1 = min(img_h,  int((y1 - pb_y0) / ch * img_h))
    return px0, py0, px1, py1


def ocr_crop(img: np.ndarray, x0: int, y0: int, x1: int, y1: int) -> str:
    """Crop a bbox from a page image and run Tesseract on it."""
    if x1 <= x0 or y1 <= y0:
        return ""
    crop = img[y0:y1, x0:x1]
    if crop.size == 0:
        return ""
    return pytesseract.image_to_string(crop, lang="eng", config=TESSERACT_CFG).strip()


def normalize(text: str) -> str:
    """Collapse whitespace for fair CER/WER comparison."""
    return re.sub(r"\s+", " ", text).strip()


print("Helper functions defined.")


## Cell 5 — Load Chandra layout blocks

In [ ]:
chandra_blocks: dict[str, list[dict]] = defaultdict(list)
skipped_label = 0

with CHANDRA_PATH.open(encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        row = json.loads(line)
        book_page = row.get("book_page")
        if book_page is None:
            continue
        label = row.get("label", "")
        if _chandra_label_kind(label) == "skip":
            skipped_label += 1
            continue   # skip Image / Figure / Diagram / Page-Header / Page-Footer
        page_id = f"p{int(book_page):04d}"
        chandra_blocks[page_id].append({
            "page_box": row.get("page_box"),   # [pb_x0, pb_y0, pb_x1, pb_y1]
            "bbox":     row.get("bbox"),        # [x0, y0, x1, y1] same pixel space
            "label":    label,
            "content":  _strip_html(row.get("content", "")),  # Chandra's text (pseudo-GT)
        })

total_pages  = len(chandra_blocks)
total_blocks = sum(len(v) for v in chandra_blocks.values())
print(f"Text blocks loaded : {total_blocks} across {total_pages} pages")
print(f"Skipped (non-text) : {skipped_label}")
print(f"Sample page IDs    : {sorted(chandra_blocks.keys())[:5]} ...")

# Quick sanity check: print first block of first page
first_pid = sorted(chandra_blocks.keys())[0]
blk = chandra_blocks[first_pid][0]
print(f"\nFirst block on {first_pid}:")
for k, v in blk.items():
    print(f"  {k}: {repr(v)[:80]}")


## Cell 6 — Load ground-truth labels

In [ ]:
gt_labels: dict[str, str] = {}

with LABELS_PATH.open(encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        row = json.loads(line)
        gt_labels[row["page_id"]] = row["text"]

print(f"Loaded {len(gt_labels)} hand-verified ground-truth pages:")
print(" ", sorted(gt_labels.keys()))

## Cell 7 — Run Tesseract (layout-masked) over all Chandra pages

> **Expected runtime**: ~1–3 s/page on Kaggle CPU = ~30–50 min for all 1,028 pages.  
> The `page_transcripts.jsonl` is saved incrementally, so you can inspect partial output if needed.

In [ ]:
doc = fitz.open(str(PDF_PATH))
print(f"PDF opened: {doc.page_count} pages")

all_page_ids = sorted(chandra_blocks.keys())
print(f"Processing {len(all_page_ids)} pages with Chandra text blocks...\n")

results: list[dict] = []
t_total = time.time()
skipped_blocks = 0

transcripts_path = OUT_DIR / "page_transcripts.jsonl"
out_f = transcripts_path.open("w", encoding="utf-8")

for page_num, page_id in enumerate(all_page_ids):
    pdf_idx = int(page_id[1:]) - 1   # 'p0024' → index 23
    if pdf_idx < 0 or pdf_idx >= doc.page_count:
        continue

    t0  = time.time()
    img = render_page(doc, pdf_idx, DPI)
    img_h, img_w = img.shape[:2]

    # Crop each text region and run Tesseract on the crop individually
    block_texts: list[str] = []
    for blk in chandra_blocks[page_id]:
        page_box = blk.get("page_box")
        bbox     = blk.get("bbox")
        if not page_box or not bbox or len(page_box) < 4 or len(bbox) < 4:
            skipped_blocks += 1
            continue
        px_result = bbox_to_pixel(bbox, page_box, img_w, img_h)
        if px_result is None:
            skipped_blocks += 1   # degenerate page_box — skip silently
            continue
        px0, py0, px1, py1 = px_result
        text = ocr_crop(img, px0, py0, px1, py1)
        if text:
            block_texts.append(text)

    page_text = "\n".join(block_texts)   # blocks in reading order
    elapsed   = time.time() - t0

    row = {"page_id": page_id, "text": page_text,
           "n_blocks": len(block_texts), "elapsed_s": round(elapsed, 2)}
    results.append(row)
    out_f.write(json.dumps(row, ensure_ascii=False) + "\n")

    if page_num % 50 == 0 or page_num < 3:
        elapsed_total = time.time() - t_total
        print(f"  [{page_num+1:4d}/{len(all_page_ids)}] {page_id} "
              f"— {len(block_texts)} blocks, {elapsed:.1f}s "
              f"(total {elapsed_total/60:.1f} min)")

out_f.close()
print(f"\nDone. {len(results)} pages in {(time.time()-t_total)/60:.1f} min.")
print(f"Skipped degenerate blocks : {skipped_blocks}")
print(f"Transcripts saved → {transcripts_path}")


## Cell 8 — Score Tesseract vs ground-truth labels (CER / WER / Word-F1)

In [ ]:
transcript_map: dict[str, str] = {r["page_id"]: r["text"] for r in results}
scored_pages: list[dict] = []

print("page_id      CER      WER   Word-F1")
print("-" * 45)

for page_id, ref_text in sorted(gt_labels.items()):
    hyp_text = transcript_map.get(page_id, "")
    ref_norm = normalize(ref_text)
    hyp_norm = normalize(hyp_text)
    if not ref_norm:
        continue

    page_cer = compute_cer(ref_norm, hyp_norm)
    page_wer = compute_wer(ref_norm, hyp_norm)

    # Set-based Word F1
    ref_words = set(ref_norm.lower().split())
    hyp_words = set(hyp_norm.lower().split())
    tp        = len(ref_words & hyp_words)
    precision = tp / len(hyp_words) if hyp_words else 0.0
    recall    = tp / len(ref_words)  if ref_words  else 0.0
    f1        = (2 * precision * recall / (precision + recall))\
                if (precision + recall) > 0 else 0.0

    scored_pages.append({
        "page_id":  page_id,
        "cer":      round(page_cer, 4),
        "wer":      round(page_wer, 4),
        "word_f1":  round(f1, 4),
        "ref_chars": len(ref_norm),
        "hyp_chars": len(hyp_norm),
    })
    print(f"{page_id}   {page_cer:.4f}   {page_wer:.4f}   {f1:.4f}")

mean_cer = mean_wer = mean_f1 = float("nan")
if scored_pages:
    mean_cer = float(np.mean([p["cer"]     for p in scored_pages]))
    mean_wer = float(np.mean([p["wer"]     for p in scored_pages]))
    mean_f1  = float(np.mean([p["word_f1"] for p in scored_pages]))
    print("-" * 45)
    print(f"MEAN         {mean_cer:.4f}   {mean_wer:.4f}   {mean_f1:.4f}")
    print(f"\n({len(scored_pages)} held-out pages scored)")

    # Save CSV
    score_path = OUT_DIR / "heldout_scores.csv"
    with score_path.open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(scored_pages[0]))
        w.writeheader(); w.writerows(scored_pages)
    print(f"Scores saved → {score_path}")

## Cell 9 — Worst failure case (for A2 form Section 5)

In [ ]:
if scored_pages:
    worst = max(scored_pages, key=lambda p: p["cer"])
    best  = min(scored_pages, key=lambda p: p["cer"])
    print(f"Best  page : {best['page_id']}  CER={best['cer']:.4f}  WER={best['wer']:.4f}")
    print(f"Worst page : {worst['page_id']}  CER={worst['cer']:.4f}  WER={worst['wer']:.4f}")

    wid = worst["page_id"]
    print(f"\n{'='*60}")
    print(f"GROUND TRUTH ({wid}) — first 500 chars:")
    print(gt_labels[wid][:500])
    print(f"\nTESSERACT OUTPUT ({wid}) — first 500 chars:")
    print(transcript_map.get(wid, "(not found)")[:500])

## Cell 10 — Compare Chandra's own text vs ground truth

In [ ]:
# Score Chandra's content field (its own OCR) against our hand-verified labels.
# This tells us: how much does Tesseract lag behind Chandra as a pseudo-GT source?

print("page_id      Chandra CER   Chandra WER")
print("-" * 42)

chandra_scores: list[tuple] = []
for page_id, ref_text in sorted(gt_labels.items()):
    chandra_text = " ".join(
        blk.get("content", "") for blk in chandra_blocks.get(page_id, [])
    )
    ref_norm = normalize(ref_text)
    hyp_norm = normalize(chandra_text)
    if not ref_norm or not hyp_norm:
        continue
    c = compute_cer(ref_norm, hyp_norm)
    w = compute_wer(ref_norm, hyp_norm)
    chandra_scores.append((page_id, c, w))
    print(f"{page_id}   {c:.4f}        {w:.4f}")

if chandra_scores:
    c_cer = float(np.mean([s[1] for s in chandra_scores]))
    c_wer = float(np.mean([s[2] for s in chandra_scores]))
    print("-" * 42)
    print(f"MEAN         {c_cer:.4f}        {c_wer:.4f}")
    print()
    print("=" * 50)
    print("COMPARISON SUMMARY")
    print("=" * 50)
    print(f"  Chandra OCR  (pseudo-GT)  :  CER={c_cer:.4f}  WER={c_wer:.4f}")
    print(f"  Tesseract 5  (layout mask):  CER={mean_cer:.4f}  WER={mean_wer:.4f}")
    delta_cer = mean_cer - c_cer
    print(f"  Gap (Tesseract − Chandra) :  ΔCER={delta_cer:+.4f}")

## Cell 11 — Write report.md

In [ ]:
lines = [
    "# Tesseract Layout-Aware OCR Benchmark — Pierce 1890",
    "",
    "## Setup",
    f"- PDF: `{PDF_PATH.name}`",
    "- Layout: Chandra `chunks.jsonl` — text blocks only (Image/Figure/Diagram excluded)",
    f"- OCR engine: Tesseract 5  lang=eng  config=`{TESSERACT_CFG}`",
    f"- Render DPI: {DPI}",
    f"- Pages processed: {len(results)} / {doc.page_count}",
    "",
    "## Results on held-out ground-truth pages (p0024–p0037)",
    "",
    "| page_id | CER | WER | Word F1 |",
    "|---|---|---|---|",
]
for p in sorted(scored_pages, key=lambda x: x["page_id"]):
    lines.append(f"| {p['page_id']} | {p['cer']:.4f} | {p['wer']:.4f} | {p['word_f1']:.4f} |")
if scored_pages:
    lines += ["", f"| **MEAN** | **{mean_cer:.4f}** | **{mean_wer:.4f}** | **{mean_f1:.4f}** |"]

lines += ["", "## Worst failure"]
if scored_pages:
    worst = max(scored_pages, key=lambda p: p["cer"])
    lines += [
        f"- Worst page: `{worst['page_id']}` — CER={worst['cer']:.4f}  WER={worst['wer']:.4f}",
        "- Cause: figure-adjacent text or header/footer noise bleeding into body crop boundaries.",
    ]

if chandra_scores:
    lines += [
        "", "## Engine comparison",
        "| Engine | Mean CER | Mean WER |",
        "|---|---|---|",
        f"| Chandra OCR (pseudo-GT reference) | {c_cer:.4f} | {c_wer:.4f} |",
        f"| Tesseract 5 (layout-masked crops)  | {mean_cer:.4f} | {mean_wer:.4f} |",
        "",
        "> Lower = better.  "
        "Chandra is the pseudo-GT; Tesseract is the fine-tuning baseline.",
    ]

report_path = OUT_DIR / "report.md"
report_path.write_text("\n".join(lines) + "\n")
print(f"Report written → {report_path}")
print()
print("\n".join(lines))

## Done — Output files

All results are in `/kaggle/working/tesseract_layout_bench/`:

| File | Contents |
|---|---|
| `page_transcripts.jsonl` | `{page_id, text, n_blocks, elapsed_s}` for every Chandra page |
| `heldout_scores.csv` | Per-page CER / WER / Word-F1 for the 14 held-out pages |
| `report.md` | Summary table + worst failure + Chandra vs Tesseract comparison |

Download them from the **Output** panel on the right.